<a href="https://colab.research.google.com/github/31shruti/Data_guard/blob/main/01_dataset_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faker pandas numpy

In [5]:
import numpy as np
import pandas as pd
from faker import Faker


In [7]:
SEED = 42
N = 8000
N_CUSTOMERS = 3000

rng = np.random.default_rng(SEED)
Faker.seed(SEED)
fake = Faker("en_IN")

CITIES = [
    "Jaipur", "Delhi", "Mumbai", "Bengaluru", "Chennai", "Kolkata",
    "Hyderabad", "Pune", "Ahmedabad", "Lucknow", "Chandigarh", "Indore",
]

PRODUCTS = {
    "Smartphone":   (15000, 80000),
    "Laptop":       (35000, 120000),
    "Smartwatch":   (2000, 25000),
    "Headphones":   (500, 8000),
    "Keyboard":     (500, 6000),
    "Mouse":        (300, 3000),
    "Desk Lamp":    (400, 3000),
    "Backpack":     (600, 4000),
    "Water Bottle": (200, 1200),
    "Notebook":     (50, 500),
}


In [9]:
customers = pd.DataFrame({
    "customer_id": [f"C{i:05d}" for i in range(1, N_CUSTOMERS + 1)],
    "customer_name": [fake.name() for _ in range(N_CUSTOMERS)],
    "email": [fake.unique.email() for _ in range(N_CUSTOMERS)],
    "city": rng.choice(CITIES, size=N_CUSTOMERS),
    "age": rng.integers(18, 81, size=N_CUSTOMERS),
})
customers.head()


,customer_id,customer_name,email,city,age
0,C00001,Nihal Parikh,geraupma@example.com,Pune,46
1,C00002,Orinder Badal,pluthra@example.com,Lucknow,19
2,C00003,Aradhana Mandal,victor96@example.com,Pune,60
3,C00004,Jatin Chaudhuri,qmore@example.org,Pune,46
4,C00005,Rohan Buch,matthew69@example.net,Kolkata,66


In [11]:
orders = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + N),
    "customer_id": rng.choice(customers["customer_id"], size=N),
})

df = orders.merge(customers, on="customer_id", how="left")
df.head()


,order_id,customer_id,customer_name,email,city,age
0,1001,C00103,Vasana Wali,shivanishanker@example.net,Kolkata,47
1,1002,C02163,Anamika Chada,lhayre@example.com,Chennai,52
2,1003,C00642,Rachit Kuruvilla,hlalla@example.com,Jaipur,63
3,1004,C02134,Nachiket Wali,ibains@example.net,Bengaluru,59
4,1005,C01694,Gagan Yadav,jhajanya@example.net,Ahmedabad,63


In [13]:
END_DATE = pd.Timestamp("2026-10-07")
days_ago = rng.integers(0, 730, size=N)
dates = END_DATE - pd.to_timedelta(days_ago, unit="D")
df["order_date"] = dates.strftime("%d/%m/%Y")

df["product"] = rng.choice(list(PRODUCTS.keys()), size=N)
df["quantity"] = rng.integers(1, 11, size=N)
df["price"] = [round(rng.uniform(*PRODUCTS[p]), 2) for p in df["product"]]

df = df[["order_id", "customer_id", "customer_name", "email", "city",
         "age", "order_date", "product", "quantity", "price"]]
df.head()


,order_id,customer_id,customer_name,email,city,age,order_date,product,quantity,price
0,1001,C00103,Vasana Wali,shivanishanker@example.net,Kolkata,47,05/08/2025,Keyboard,1,4135.93
1,1002,C02163,Anamika Chada,lhayre@example.com,Chennai,52,18/08/2025,Keyboard,2,4760.29
2,1003,C00642,Rachit Kuruvilla,hlalla@example.com,Jaipur,63,01/09/2025,Laptop,1,67601.13
3,1004,C02134,Nachiket Wali,ibains@example.net,Bengaluru,59,02/03/2025,Notebook,5,353.60
4,1005,C01694,Gagan Yadav,jhajanya@example.net,Ahmedabad,63,18/10/2024,Headphones,1,7272.35


In [15]:
assert len(df) == 8000
assert df.isna().sum().sum() == 0
assert df["order_id"].is_unique
assert df["age"].between(18, 80).all()
assert df["quantity"].between(1, 10).all()
assert set(df["city"]) == set(CITIES)
print("All checks passed")

df.describe()


All checks passed


,order_id,age,quantity,price
count,8000.00000,8000.000000,8000.000000,8000.000000
mean,5000.50000,49.013250,5.495125,14969.978909
std,2309.54541,17.850032,2.875458,26623.388795
min,1001.00000,18.000000,1.000000,50.170000
25%,3000.75000,34.000000,3.000000,957.580000
50%,5000.50000,49.000000,5.000000,2567.705000
75%,7000.25000,64.000000,8.000000,11654.100000
max,9000.00000,80.000000,10.000000,119978.750000


In [17]:
df.to_csv("clean_orders.csv", index=False)

In [19]:
RATES = {
    "missing_age": 0.04,
    "missing_product": 0.02,
    "duplicate_order_id": 0.01,
    "invalid_email": 0.02,
    "negative_quantity": 0.005,
    "impossible_age": 0.005,
    "city_variant": 0.05,
    "mixed_date_format": 0.03,
    "price_outlier": 0.005,
}
DUPLICATE_ROW_RATE = 0.01

shuffled = rng.permutation(N)
rows_for = {}
pos = 0
for ctype, rate in RATES.items():
    k = int(N * rate)
    rows_for[ctype] = shuffled[pos:pos + k]
    pos += k

k_dup = int(N * DUPLICATE_ROW_RATE)
dup_source_rows = shuffled[pos:pos + k_dup]

print({c: len(r) for c, r in rows_for.items()}, k_dup)


{'missing_age': 320, 'missing_product': 160, 'duplicate_order_id': 80, 'invalid_email': 160, 'negative_quantity': 40, 'impossible_age': 40, 'city_variant': 400, 'mixed_date_format': 240, 'price_outlier': 40} 80


In [21]:
dirty = df.copy()
dirty["age"] = dirty["age"].astype("Int64")
log = []

def record(row_id, column, ctype, old, new):
    log.append({
        "row_id": int(row_id),
        "column": column,
        "corruption_type": ctype,
        "original_value": old,
        "corrupted_value": new,
    })


In [23]:
# 1. Missing ages
for r in rows_for["missing_age"]:
    record(r, "age", "missing_age", dirty.at[r, "age"], None)
    dirty.at[r, "age"] = pd.NA

# 2. Missing products (half empty, half whitespace-only)
for i, r in enumerate(rows_for["missing_product"]):
    new = np.nan if i % 2 == 0 else "  "
    record(r, "product", "missing_product", dirty.at[r, "product"], new)
    dirty.at[r, "product"] = new

# 3. Duplicate order IDs (copy another row's ID)
candidates = np.setdiff1d(np.arange(N), rows_for["duplicate_order_id"])
for r in rows_for["duplicate_order_id"]:
    src = rng.choice(candidates)
    new_id = dirty.at[src, "order_id"]
    record(r, "order_id", "duplicate_order_id", dirty.at[r, "order_id"], new_id)
    dirty.at[r, "order_id"] = new_id

# 4. Invalid emails
def break_email(email):
    user, domain = email.split("@")
    m = rng.integers(0, 4)
    if m == 0:
        return user + domain          # missing @
    if m == 1:
        return user + "@"             # missing domain
    if m == 2:
        return user + " @" + domain   # space inside
    return user + "@" + domain.split(".")[0]   # missing .com / .in

for r in rows_for["invalid_email"]:
    old = dirty.at[r, "email"]
    new = break_email(old)
    record(r, "email", "invalid_email", old, new)
    dirty.at[r, "email"] = new

# 5. Negative quantity
for r in rows_for["negative_quantity"]:
    old = dirty.at[r, "quantity"]
    record(r, "quantity", "negative_quantity", old, -old)
    dirty.at[r, "quantity"] = -old

# 6. Impossible ages
for r in rows_for["impossible_age"]:
    new = int(rng.choice([-5, 0, 150, 999]))
    record(r, "age", "impossible_age", dirty.at[r, "age"], new)
    dirty.at[r, "age"] = new

# 7. City variants
def vary_city(city):
    m = rng.integers(0, 4)
    if m == 0:
        return city.lower()
    if m == 1:
        return city.upper()
    if m == 2:
        return city + " "
    return city[:2] + city[3:]        # typo: drop 3rd letter

for r in rows_for["city_variant"]:
    old = dirty.at[r, "city"]
    new = vary_city(old)
    record(r, "city", "city_variant", old, new)
    dirty.at[r, "city"] = new

# 8. Mixed date formats
DATE_FORMATS = ["%b %d, %Y", "%Y-%m-%d", "%d-%m-%Y"]
for r in rows_for["mixed_date_format"]:
    old = dirty.at[r, "order_date"]
    parsed = pd.to_datetime(old, format="%d/%m/%Y")
    new = parsed.strftime(DATE_FORMATS[rng.integers(0, 3)])
    record(r, "order_date", "mixed_date_format", old, new)
    dirty.at[r, "order_date"] = new

# 9. Price outliers
for r in rows_for["price_outlier"]:
    old = dirty.at[r, "price"]
    new = round(old * rng.uniform(50, 100), 2)
    record(r, "price", "price_outlier", old, new)
    dirty.at[r, "price"] = new


ValueError: not enough values to unpack (expected 2, got 1)

In [25]:
copies = dirty.loc[dup_source_rows].copy()
start = len(dirty)
for i, src in enumerate(dup_source_rows):
    record(start + i, "ALL", "duplicate_row", int(src), "exact copy")

dirty = pd.concat([dirty, copies], ignore_index=True)


In [26]:
ground_truth = pd.DataFrame(log)

print(dirty.shape)                      # expect 8080 rows
print(ground_truth["corruption_type"].value_counts())
print("Total injected:", len(ground_truth))

dirty.to_csv("dirty_orders.csv", index=False)
ground_truth.to_csv("ground_truth.csv", index=False)


(8160, 10)
corruption_type
missing_age           640
city_variant          400
missing_product       320
mixed_date_format     240
invalid_email         163
duplicate_order_id    160
duplicate_row         160
negative_quantity      40
impossible_age         40
price_outlier          40
Name: count, dtype: int64
Total injected: 2203
